In [59]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer

In [60]:
cancer = load_breast_cancer()

In [61]:
df = pd.DataFrame(data = cancer.data, columns = cancer.feature_names)
df['target'] = cancer.target

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [66]:

X = df.drop('target', axis=1)
y = df['target']
df.shape


(569, 31)

In [72]:
from sklearn.model_selection import train_test_split
X_train_90, X_test_10, y_train_90, y_test_10 = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)


In [73]:
X_dtrain, X_dval, y_dtrain, y_dval = train_test_split(
    X_train_90, y_train_90, test_size=0.20, random_state=42, stratify=y_train_90
)

print(f"1. Main Test Set (10%): {len(X_test_10)} rows  <-- (Ise abhi touch nahi karenge)")
print(f"2. Base Model Training Data (d_train): {len(X_dtrain)} rows")
print(f"3. Meta Model Data Creation Set (d_val): {len(X_dval)} rows\n")

1. Main Test Set (10%): 57 rows  <-- (Ise abhi touch nahi karenge)
2. Base Model Training Data (d_train): 409 rows
3. Meta Model Data Creation Set (d_val): 103 rows



In [74]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [75]:
m1 = DecisionTreeClassifier(random_state=42)
m2 = LogisticRegression(max_iter=5000, random_state=42)
m3 = KNeighborsClassifier()

pred1 = m1.fit(X_dtrain, y_dtrain)
pred2 = m2.fit(X_dtrain, y_dtrain)
pred3 = m3.fit(X_dtrain, y_dtrain)

In [78]:
# 2. Naya Table (Meta Dataframe) banana
meta_df = pd.DataFrame({
    'm1_pred': pred1,
    'm2_pred': pred2,
    'm3_pred': pred3,
    'target': y
})

# Naye table ki pehli 10 rows dekhne ke liye
meta_df.shape

(569, 4)

In [81]:
# 1. Teeno Base Models se 'X_dval' par predictions nikalna
val_pred_1 = m1.predict(X_dval)
val_pred_2 = m2.predict(X_dval)
val_pred_3 = m3.predict(X_dval)

# 2. In predictions se Meta-Model ke liye DataFrame (Table) banana
meta_X_train = pd.DataFrame({
    'pred_m1': val_pred_1,
    'pred_m2': val_pred_2,
    'pred_m3': val_pred_3
})

# Meta dataset ko check karna (Isme sirf 3 columns honge)
print("Meta-Model ke liye bana dataset:")
meta_X_train.shape

Meta-Model ke liye bana dataset:


(103, 3)

In [82]:
# 1. Meta Model choose kiya (Jaise Logistic Regression ya Random Forest)
meta_model = LogisticRegression()

# 2. Meta Model ko Meta-Data par train/fit karna
meta_model.fit(meta_X_train, y_dval)

print("🎉 Meta-Model successfully train ho gaya hai!")

🎉 Meta-Model successfully train ho gaya hai!


In [89]:
X_test_10 = X_test_10.reset_index(drop=True)
y_test_10 = y_test_10.reset_index(drop=True)
X_test_10.head()


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,14.45,20.22,94.49,642.7,0.09872,0.1206,0.11800,0.05980,0.1950,0.06466,...,18.33,30.12,117.90,1044.0,0.1552,0.4056,0.4967,0.18380,0.4753,0.10130
1,11.26,19.96,73.72,394.1,0.08020,0.1181,0.09274,0.05588,0.2595,0.06233,...,11.86,22.33,78.27,437.6,0.1028,0.1843,0.1546,0.09314,0.2955,0.07009
2,13.80,15.79,90.43,584.1,0.10070,0.1280,0.07789,0.05069,0.1662,0.06566,...,16.57,20.86,110.30,812.4,0.1411,0.3542,0.2779,0.13830,0.2589,0.10300
3,18.66,17.12,121.40,1077.0,0.10540,0.1100,0.14570,0.08665,0.1966,0.06213,...,22.25,24.90,145.40,1549.0,0.1503,0.2291,0.3272,0.16740,0.2894,0.08456
4,13.73,22.61,93.60,578.3,0.11310,0.2293,0.21280,0.08025,0.2069,0.07682,...,15.03,32.01,108.80,697.7,0.1651,0.7725,0.6943,0.22080,0.3596,0.14310


In [98]:
y_test_10.head(10)


,target
0,0
1,1
2,0
3,0
4,0
5,1
6,0
7,1
8,1
9,1


In [99]:
single_row = X_test_10.iloc[[7]]

In [100]:
p1 = m1.predict(single_row)[0]
p2 = m2.predict(single_row)[0]
p3 = m3.predict(single_row)[0]

print(f"Model 1 Prediction: {p1}")
print(f"Model 2 Prediction: {p2}")
print(f"Model 3 Prediction: {p3}")

Model 1 Prediction: 1
Model 2 Prediction: 1
Model 3 Prediction: 1


In [102]:
# 3. Meta-Model ke liye input DataFrame banana
meta_single_input = pd.DataFrame({
    'pred_m1': [p1],
    'pred_m2': [p2],
    'pred_m3': [p3]
})

# 4. Meta-Model se final output
final_pred = meta_model.predict(meta_single_input)[0]

# 5. Actual Result se compare karna
actual_val = y_test_10.iloc[7]

print("\n--------------------------------")
print(f"🎉 Meta-Model Final Prediction: {final_pred}")
print(f"✅ Actual Value in Data      : {actual_val}")
print("--------------------------------")


--------------------------------
🎉 Meta-Model Final Prediction: 1
✅ Actual Value in Data      : 1
--------------------------------
